# CareLoop Clinics — Part 4: GenAI Follow-Up Review
Run this notebook in Google Colab. Enter the Gemini key only at runtime. Never save or upload the key.

In [ ]:
!pip install -q google-genai pydantic

In [ ]:
import json, time
import pandas as pd
from getpass import getpass
from google import genai
from google.genai import types
from pydantic import BaseModel, Field, ValidationError, field_validator
API_KEY = getpass("Paste your Gemini API key: ")
client = genai.Client(api_key=API_KEY)
MODEL_NAME = "gemini-2.5-flash"  # If retired, select the current Flash model shown in AI Studio.
CALL_COUNT = 0

## Task 4.1 — One validated call

In [ ]:
ALLOWED = ["forgot_or_no_reminder","distance_or_transport","cost_or_fee",
           "illness_or_emergency","work_or_schedule","other"]
SCHEMA = {"type":"object","properties":{
 "root_cause":{"type":"string"},"confidence":{"type":"number"},
 "summary":{"type":"string"},"recommended_action":{"type":"string"}},
 "required":["root_cause","confidence","summary","recommended_action"]}
SYSTEM_TEXT = "You classify outpatient missed-appointment follow-up notes. Return only valid JSON matching the schema."

def craft_prompt(note_text):
    return f"""Context: CareLoop is reviewing why a patient missed an appointment.
Role: You are a careful healthcare operations analyst.
Action: Classify the note into exactly one allowed root cause: {ALLOWED}. Do not invent facts. If ambiguous, use other.
Tone: Concise and neutral.
Format: Return root_cause, confidence from 0 to 1, a summary of at most 25 words, and a practical recommended_action.
Patient note: {note_text}"""

def call_model(prompt):
    global CALL_COUNT
    CALL_COUNT += 1
    config = types.GenerateContentConfig(system_instruction=SYSTEM_TEXT, temperature=0,
        max_output_tokens=1000, response_mime_type="application/json", response_schema=SCHEMA,
        thinking_config=types.ThinkingConfig(thinking_level="minimal"))
    try:
        return client.models.generate_content(model=MODEL_NAME, contents=prompt, config=config)
    except Exception as exc:
        if "thinking" not in str(exc).lower(): raise
        config = types.GenerateContentConfig(system_instruction=SYSTEM_TEXT, temperature=0,
            max_output_tokens=1000, response_mime_type="application/json", response_schema=SCHEMA)
        return client.models.generate_content(model=MODEL_NAME, contents=prompt, config=config)

def analyse_note(note_text):
    response = call_model(craft_prompt(note_text))
    return json.loads(response.text), response.text

notes = pd.read_csv("data/followup_notes.csv")
first, raw = analyse_note(notes.loc[notes["note_id"]=="N001","note_text"].iloc[0])
print(first)

## Task 4.2 — Validate and process the first 10 notes

In [ ]:
class FollowupInsight(BaseModel):
    root_cause: str = Field(min_length=3)
    confidence: float = Field(ge=0, le=1)
    summary: str = Field(min_length=5)
    recommended_action: str = Field(min_length=5)
    @field_validator("root_cause")
    @classmethod
    def allowed_root_cause(cls, value):
        if value not in ALLOWED: raise ValueError("root_cause is not in the allowed list")
        return value

records=[]
for _,row in notes.head(10).iterrows():
    note_text=row["note_text"]
    if pd.isna(note_text) or str(note_text).strip()=="":
        records.append({"note_id":row.note_id,"appointment_id":row.appointment_id,
                        "root_cause":"other","confidence":0.0,"summary":"No note recorded",
                        "recommended_action":"Re-call patient","validation_status":"skipped_blank","raw_response":""})
        continue
    raw_response=""; data={}
    try:
        data, raw_response = analyse_note(str(note_text))
        FollowupInsight(**data)
        status="valid"
    except Exception as exc:
        status="invalid"
        if not raw_response: raw_response=str(exc)
    records.append({"note_id":row.note_id,"appointment_id":row.appointment_id,
                    "root_cause":data.get("root_cause"),"confidence":data.get("confidence"),
                    "summary":data.get("summary"),"recommended_action":data.get("recommended_action"),
                    "validation_status":status,"raw_response":raw_response})
    time.sleep(13)
analysed=pd.DataFrame(records)
analysed.to_csv("notes_analysed.csv",index=False)
print(analysed[["note_id","root_cause","confidence","validation_status"]])

## Task 4.3 — Break validation and compare prompts

In [ ]:
bad_replies=[
 {"root_cause":"other","confidence":1.7,"summary":"Ambiguous patient explanation","recommended_action":"Call patient again"},
 {"root_cause":"other","confidence":0.4,"summary":"Ambiguous patient explanation"}]
for bad in bad_replies:
    try: FollowupInsight(**bad)
    except ValidationError as exc: print(exc)

def v1_prompt(note): return f"Classify this note: {note}"
comparison=[]
for note_id in ["N007","N009"]:
    note=notes.loc[notes.note_id==note_id,"note_text"].iloc[0]
    for version,prompt in [("V1",v1_prompt(note)),("V2",craft_prompt(note))]:
        response=call_model(prompt); data=json.loads(response.text)
        comparison.append({"note_id":note_id,"prompt":version,"root_cause":data.get("root_cause"),
                           "allowed":data.get("root_cause") in ALLOWED})
        time.sleep(13)
print(pd.DataFrame(comparison))
print("Deploy V2 because it constrains labels and instructs the model to use other rather than guess.")
print("Total API calls:",CALL_COUNT)

## Task 4.4 — Merge, cross-tabulate and report

In [ ]:
appointments=pd.read_csv("appointments_clean.csv")
analysed=pd.read_csv("notes_analysed.csv")
merged=analysed.merge(appointments,on="appointment_id",how="left")
print("Merged rows:",len(merged))
print(pd.crosstab(merged["root_cause"],merged["reminder_sent"]))
print("Use these actual results to update README.md before submission.")